<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

## SEC Executive Extraction Pipeline

## Project Overview
**Task**: Extract executive information (names and titles) from SEC filings  
**Dataset**: 3.3GB CSV file containing SEC documents (8-K, 10-K, DEF 14A, S-1, etc.)  
**Goal**: Build a proprietary database of company executives from official SEC filings

---

## What This Pipeline Does?
1. **Test Mode**: Analyzes first 50 rows to validate extraction patterns
2. **Full Processing**: Extracts executives from entire dataset in chunks
3. **Cleaning**: Deduplicates and normalizes the extracted data
4. **Analysis**: Generates quality metrics and insights

---

## Libraries Used

- **Packages**: `pandas`, `psutil`, `tqdm`, `spaCy`

---


## Quick Start

### Step 1: Configure
```
INPUT_FILE = "8k_filings_raw_text_2024.csv"
OUTPUT_RAW = "executives_raw.csv"
OUTPUT_CLEAN = "executives_final.csv"
```

### Step 2: Run Pipeline
```
python GUU_Task_1.py
```

### Step 3: Review Results
- Check test output (first 50 rows)
- Confirm to proceed with full extraction
- Review final output: `executives_final.csv`

---

## 🔍 Extraction Strategy

The pipeline uses **3 pattern-matching strategies** with confidence levels:

### 1️⃣ High Confidence
```
By: /s/ Brandon Ehrhart
Brandon Ehrhart
General Counsel and Corporate Secretary
```

### 2️⃣ Medium Confidence
```
By: /s/ Marc D. Hamburg
[searches nearby text for title]
```

### 3️⃣ Low Confidence
```
"John Smith, Chief Executive Officer"
```

---

## 🐛 Troubleshooting

| Issue | Solution |
|-------|----------|
| Out of memory | Reduce `CHUNK_SIZE` to 5000 |
| Too slow | Increase `CHUNK_SIZE` to 20000 |
| Low hit rate (<30%) | Check test output, adjust patterns |
| Script crashes | Check CSV encoding, try `encoding='utf-8'` |


In [1]:
"""
SEC Executive Extraction Pipeline
Author: David Acevedo-Cardona
"""

#Imports:
import pandas as pd
import re, csv, spacy
from pathlib import Path
from datetime import datetime
from typing import List, Tuple
from collections import Counter

In [3]:
#Print some values:

filings_df = "8k_filings_raw_text_2024.csv"

# Display some rows
df_sample = pd.read_csv(filings_df, nrows=5)

#Print
df_sample

,sec_accession_number,release_datetime,title,sec_filing_type,keywords,exchange,symbol,company_name,excerpt,raw_text
0,0000950170-24-000282,2024-01-03 08:29:17+11:00,Tesla Reports Record Vehicle Production and De...,8-K,"Tesla,Vehicle Production,Vehicle Deliveries,Fi...",NASDAQ,TSLA,"Tesla, Inc.",Tesla announced record vehicle production and ...,=== MAIN 8-K FILING ===\n8-K false-01-022024-0...
1,0001193125-24-005943,2024-01-11 08:16:32+11:00,Berkshire Hathaway Settles Delaware Litigation...,8-K,"litigation,settlement,Berkshire Hathaway,Pilot...",NYSE,BRK-B,BERKSHIRE HATHAWAY INC,Berkshire Hathaway has reached a full settleme...,=== MAIN 8-K FILING ===\n8-K BERKSHIRE HATHAWA...
2,0001213900-24-002759,2024-01-11 09:44:11+11:00,Steel Partners Holdings L.P. Abandons Previous...,8-K,"unit split,reverse split,forward split,share r...",NYSE,SPLP,STEEL PARTNERS HOLDINGS L.P.,Steel Partners Holdings L.P. has announced the...,=== MAIN 8-K FILING ===\nfalse 2024-01-10 20...
3,0001213900-24-002761,2024-01-11 09:45:04+11:00,Akerna Sets Special Meeting Date for Gryphon D...,8-K,"Merger,Akerna,Gryphon Digital Mining,Stockhold...",NASDAQ,GRYP,"Gryphon Digital Mining, Inc.",Akerna has announced the date for a special st...,=== MAIN 8-K FILING ===\nfalse 2024-01-10 20...
4,0001683168-24-000184,2024-01-11 09:47:10+11:00,"Focus Universal Secures $300,000 Loan as Part ...",8-K,"revolving credit facility,loan,financing,debt,...",NASDAQ,FCUV,FOCUS UNIVERSAL INC.,"Focus Universal Inc. has entered into a $300,0...",=== MAIN 8-K FILING ===\nfalse 2024-01-09 20...


In [ ]:
#Define the classes:

class ExecutiveExtractor:
    """
    Extract executive information from SEC filing text
    
    """

    def __init__(self):
        
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "tagger"]) # Load spaCy
        
        # Common executive titles
        self.exec_titles = [
            'CEO', 'CFO', 'COO', 'CTO', 'CLO',
            'Chief Executive Officer', 'Chief Financial Officer',
            'Chief Operating Officer', 'Chief Technology Officer',
            'President', 'Vice President', 'Senior Vice President',
            'General Counsel', 'Corporate Secretary',
            'Chairman', 'Director', 'Treasurer'
        ]

    # Main extraction function
    def extract_executives(self, text: str) -> List[Tuple[str, str, str]]:
        """
        Merge the regex with spaCy filters
        
        """
        regex_ans = self._extract_via_regex(text)
        spacy_ans = self._extract_via_spacy(text)
        merge = regex_ans + spacy_ans
        return self._deduplicate_executives(merge)


    # Regex extraction
    def regex_extract(self, text: str) -> List[Tuple[str, str, str]]:
        
        #Empty list
        executives = []

        # Signature blocks: 
        regex1 = re.compile(
            r'By:\s*/s/\s*([A-Z][a-zA-Z\.\s]+?)\s*\n\s*(?:Name:\s*)?([A-Z][a-zA-Z\.\s]+?)\s*\n\s*(?:Title:\s*)?([^\n]+?)(?=\n\s*Date:|\n\n|\Z)',
            re.MULTILINE
        )
        
        for match in regex1.finditer(text):
            name = match.group(2).strip()
            title = re.sub(r'\s*Date:.*', '', match.group(3)).strip()
            if len(name.split()) >= 2 and self._is_executive_title(title):
                executives.append((name, title, 'high'))

        # Signature with title nearby
        pattern2 = re.compile(r'By:\s*/s/\s*([A-Z][a-zA-Z\.\s]+?)', re.MULTILINE)
        for match in pattern2.finditer(text):
            name = match.group(1).strip()
            next_text = text[match.end():match.end() + 200]
            title_match = re.search(
                r'([A-Z][a-zA-Z\s,&]+(?:Officer|President|Director|Counsel|Secretary|Chairman|Treasurer))',
                next_text
            )
            if title_match and len(name.split()) >= 2:
                title = title_match.group(1).strip()
                executives.append((name, title, 'medium'))

        # Pattern 3: Direct mentions like "John Smith, CEO"
        pattern3 = re.compile(
            r'([A-Z][a-z]+(?:\s+[A-Z]\.?)?\s+[A-Z][a-z]+),\s*(Chief\s+\w+\s+Officer|President|Vice\s+President|General\s+Counsel|Director)',
            re.IGNORECASE
        )
        for match in pattern3.finditer(text):
            name = match.group(1).strip()
            title = match.group(2).strip()
            if len(name.split()) >= 2 and self._is_executive_title(title):
                executives.append((name, title, 'low'))

        return executives


    # ========== 1C. spaCy-based extraction ==========
    def _extract_via_spacy(self, text: str) -> List[Tuple[str, str, str]]:
        """Identify PERSON entities near executive titles"""
        doc = self.nlp(text)
        executives = []
        exec_keywords = ['chief', 'officer', 'president', 'counsel', 'secretary', 'chairman', 'treasurer', 'director']

        for ent in doc.ents:
            if ent.label_ == "PERSON":
                context_window = text[max(0, ent.start_char - 80): ent.end_char + 80].lower()
                for kw in exec_keywords:
                    if kw in context_window:
                        title_match = re.search(
                            r'(chief\s+[a-z]+\s+officer|president|vice\s+president|general\s+counsel|corporate\s+secretary|chairman|director)',
                            context_window, re.IGNORECASE)
                        title = title_match.group(1).title() if title_match else 'Unknown'
                        executives.append((ent.text.strip(), title.strip(), 'spacy'))
                        break
        return executives


    # ========== 1D. Helpers ==========
    def _is_executive_title(self, title: str) -> bool:
        keywords = ['chief', 'officer', 'president', 'counsel', 'secretary', 'chairman', 'director', 'treasurer']
        return any(k in title.lower() for k in keywords)

    def _deduplicate_executives(self, executives: List[Tuple]) -> List[Tuple]]:
        if not executives:
            return []
        confidence_order = {'high': 0, 'medium': 1, 'spacy': 2, 'low': 3}
        name_dict = {}
        for name, title, conf in executives:
            key = name.lower().strip()
            if key not in name_dict or confidence_order[conf] < confidence_order[name_dict[key][2]]:
                name_dict[key] = (name, title, conf)
        return list(name_dict.values())
